# Limpieza y Análisis Exploratorio de Datos de Entregas - LogiTech Distribution

En este notebook limpiamos el dataset generado con errores simulados y extraemos los primeros hallazgos.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cargar datos sucios (ajusta la ruta si es necesario)
df = pd.read_csv('../data/entregas_sucias.csv')
df.head()

In [ ]:
df.info()
df.isnull().sum()

In [ ]:
**Problemas detectados:**
- La columna `fecha_salida` contiene algunas fechas en formato `dd/mm/yyyy` (string) en lugar de datetime.
- Existen valores nulos en `dias_reales` (entregas no registradas).
- Los nombres de productos están en minúsculas y con guiones bajos.
- No hay coordenadas erróneas (verificaremos).

In [ ]:
from datetime import datetime

def parse_fecha(fecha):
    try:
        # Si ya es datetime, se devuelve igual (solo ocurre si no es string)
        if isinstance(fecha, float):  # NaN
            return pd.NaT
        return pd.to_datetime(fecha)
    except:
        # Si falla, es porque está en formato día/mes/año
        return datetime.strptime(fecha, '%d/%m/%Y')

df['fecha_salida'] = df['fecha_salida'].apply(parse_fecha)
df['fecha_salida'].head()

In [ ]:
# Supuesto: si no se registró la entrega, asumimos que llegó con un día extra de retraso
df['dias_reales'] = df['dias_reales'].fillna(df['dias_previstos'] + 1)
df.isnull().sum()  # Verificamos que no queden nulos

In [ ]:
df['producto'] = df['producto'].str.replace('_', ' ').str.title()
df['producto'].unique()

In [ ]:
df['margen_neto'] = df['ingreso_venta'] - df['costo_envio'] - df['penalizacion']

In [ ]:
# Tasa de retraso por país
retraso_pais = df.groupby('pais')['estado'].apply(lambda x: (x == 'Retrasado').mean() * 100)
print(retraso_pais)

# Margen promedio por producto
margen_prod = df.groupby('producto')['margen_neto'].mean().sort_values()
print(margen_prod)

In [ ]:
df.to_csv('../data/entregas_limpias.csv', index=False)
print('Datos limpios guardados.')